# Chapter 13: Debugging Workflows That Didn't Fire (Reference)

## Learning Objectives

- Diagnose a YAML syntax error and report its line
- Replay paths: filter matching against a changed-file list
- Check whether a fired event type is declared under on:
- Recite the ordered diagnostic checklist

## Setup

The next cell sets up reproducibility and the `PRA_MODE` toggle. You should see `PRA_MODE = 'fixture'` printed by default.

In [1]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")

PRA_MODE = 'fixture'


## 1. YAML Syntax Diagnosis

The next cell checks a deliberately broken YAML snippet, then a well-formed one. You should see `valid=False` with a line number for the first, `valid=True` for the second.

In [2]:
from labs.lab_13_why_no_trigger import diagnose_yaml_syntax

broken = "on:\n  pull_request:\n  paths:\n   - 'sandbox/**'\n  bad indent here"
result = diagnose_yaml_syntax(broken)
print(f"valid={result['valid']}, line={result['line']}, error={result['error']}")

good = "on:\n  pull_request:\n    paths:\n      - 'sandbox/**'\n"
print(f"valid={diagnose_yaml_syntax(good)['valid']} (well-formed example)")

valid=False, line=5, error=while scanning a simple key
valid=True (well-formed example)


## 2. Path-Filter Replay

The next cell replays `paths: ['sandbox/**']` against two different changed-file sets. You should see the curriculum-only PR correctly NOT match, and the sandbox PR correctly match.

In [3]:
from labs.lab_13_why_no_trigger import check_path_filter_match

curriculum_pr = ["learning_modules/chapter_13_debugging_workflows.md"]
sandbox_pr = ["sandbox/generated/pr_120.py"]
print(
    f"curriculum PR matches sandbox/**? {check_path_filter_match(curriculum_pr, ['sandbox/**'])}"
)
print(
    f"sandbox PR matches sandbox/**?     {check_path_filter_match(sandbox_pr, ['sandbox/**'])}"
)

curriculum PR matches sandbox/**? False
sandbox PR matches sandbox/**?     True


## 3. Event-Type Matching

The next cell checks three candidate fired events against a sample `on:` block. You should see `pull_request` and `workflow_dispatch` match, `push` not match.

In [4]:
from labs.lab_13_why_no_trigger import check_event_type_match

on_block = {"pull_request": {"paths": ["sandbox/**"]}, "workflow_dispatch": {}}
for event in ["pull_request", "push", "workflow_dispatch"]:
    print(
        f"fired={event!r} matches on: block? {check_event_type_match(event, on_block)}"
    )

fired='pull_request' matches on: block? True
fired='push' matches on: block? False
fired='workflow_dispatch' matches on: block? True


## Takeaways & Next Steps

This notebook's takeaway is the path-filter replay above -- a curriculum-only PR correctly triggering zero gates is this repo's design working as intended, not a bug.

In [5]:
from labs.lab_13_why_no_trigger import DIAGNOSTIC_CHECKLIST

for i, item in enumerate(DIAGNOSTIC_CHECKLIST, start=1):
    print(f"[{i}] {item}")

[1] Does the workflow file parse as valid YAML at all?
[2] Is the workflow file on the DEFAULT branch? (schedule/workflow_dispatch need this)
[3] Does the fired event type match anything under on:?
[4] If on: has a paths: filter, does the changed-file set actually match a glob?
[5] If on: has a branches: filter, does the target branch match?
[6] Is the workflow disabled in the repo's Actions settings?
[7] Is this a fork PR hitting a restriction on secrets/workflow runs?


---

📖 **Reading companion:** [Chapter 13: Debugging Workflows That Didn't Fire](../learning_modules/chapter_13_debugging_workflows.md)
